In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import statsmodels.api as sm

from ydata_profiling import ProfileReport
import pygwalker as pyg


In [11]:
df = pd.read_csv('data/eurostat_migr_eipre_uk.csv')
profile = ProfileReport(df, title="YData Profiling Report")

In [12]:
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 60.49it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
walker = pyg.walk(df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00064f68516e9d59CJklBHVxhta86nN4" style="height: auto">\n    <hea…

In [14]:
import pandas as pd
import plotly.graph_objects as go
import math

# --- Country centroid coordinates (lat, lon) ---
COORDS = {
    'SY': (35.0, 38.0),    'AF': (33.9, 67.7),    'TN': (34.0, 9.5),
    'PK': (30.4, 69.3),    'EG': (26.8, 30.8),    'TR': (39.9, 32.9),
    'BD': (23.7, 90.4),    'IQ': (33.2, 43.7),    'IN': (20.6, 78.9),
    'MA': (31.8, -7.1),    'AL': (41.2, 20.2),    'BI': (-3.4, 29.9),
    'IR': (32.4, 53.7),    'CI': (7.5, -5.5),     'GN': (9.9, -9.7),
    'NG': (9.1, 8.7),      'CN': (35.9, 104.2),   'BR': (-14.2, -51.9),
    'GH': (7.9, -1.0),     'VN': (14.1, 108.3),   'UA': (48.4, 31.2),
    'SD': (12.9, 30.2),    'LK': (7.9, 80.8),     'JM': (18.1, -77.3),
    'ER': (15.2, 39.8),    'DZ': (28.0, 1.7),     'PH': (12.9, 121.8),
    'NP': (28.4, 84.1),    'SO': (5.2, 46.2),     'PS': (31.9, 35.2),
    'GE': (42.3, 43.4),    'UK': (55.4, -3.4),
}

# --- Load EU27 illegal entry data ---
df_eu = pd.read_csv('data/eurostat_migr_eipre_eu27.csv')
entry = df_eu[
    (df_eu['reason_code'] == 'ENTRY_NLEG') &
    (~df_eu['citizen_code'].isin(['TOTAL', 'UNK'])) &
    (df_eu['time_code'].isin([2022, 2023, 2024]))
]
flows = entry.groupby(['citizen_code', 'citizen_label'])['value'].sum().reset_index()
flows = flows.sort_values('value', ascending=False).head(15)
flows = flows[flows['citizen_code'].isin(COORDS)]

# Destination: centre of EU (roughly Brussels)
DST_LAT, DST_LON = 50.8, 4.4

# --- Build the figure ---
fig = go.Figure()

# Choropleth underlay: shade source countries by flow volume
fig.add_trace(go.Choropleth(
    locations=flows['citizen_code'],
    z=flows['value'],
    colorscale='YlOrRd',
    showscale=True,
    colorbar=dict(
        title=dict(text='Persons<br>(2022–24)', font=dict(size=11)),
        len=0.5, y=0.3, thickness=12,
    ),
    marker_line_color='#444',
    marker_line_width=0.5,
    hovertemplate='%{text}<br>%{z:,.0f} persons<extra></extra>',
    text=flows['citizen_label'],
))

max_val = flows['value'].max()

for _, row in flows.iterrows():
    code = row['citizen_code']
    src_lat, src_lon = COORDS[code]
    value = row['value']
    width = max(1.0, 8 * (value / max_val))
    opacity = max(0.35, 0.9 * (value / max_val))

    fig.add_trace(go.Scattergeo(
        lon=[src_lon, DST_LON],
        lat=[src_lat, DST_LAT],
        mode='lines',
        line=dict(width=width, color=f'rgba(180, 30, 30, {opacity})'),
        name=row['citizen_label'],
        hoverinfo='text',
        text=f"{row['citizen_label']}: {value:,.0f}",
        showlegend=False,
    ))

    # Source country marker
    fig.add_trace(go.Scattergeo(
        lon=[src_lon], lat=[src_lat],
        mode='markers+text',
        marker=dict(size=max(5, 14 * (value / max_val)), color='darkred', opacity=0.8),
        text=code,
        textposition='top center',
        textfont=dict(size=9, color='#333'),
        hoverinfo='text',
        hovertext=f"{row['citizen_label']}<br>{value:,.0f} illegal entries (2022–24)",
        showlegend=False,
    ))

# Destination marker (EU)
fig.add_trace(go.Scattergeo(
    lon=[DST_LON], lat=[DST_LAT],
    mode='markers+text',
    marker=dict(size=16, color='#003399', symbol='star', line=dict(width=1, color='white')),
    text=['EU'],
    textposition='bottom center',
    textfont=dict(size=11, color='#003399', family='Arial Black'),
    hoverinfo='text',
    hovertext='European Union',
    showlegend=False,
))

fig.update_layout(
    title=dict(
        text='Illegal Entry to the EU by Nationality (2022–2024)',
        font=dict(size=18, family='Arial'),
        x=0.5,
    ),
    geo=dict(
        showcountries=True,
        countrycolor='#bbb',
        showland=True,
        landcolor='#f0f0f0',
        showocean=True,
        oceancolor='#e8f0fe',
        showlakes=True,
        lakecolor='#e8f0fe',
        projection_type='natural earth',
        center=dict(lat=30, lon=35),
        projection_scale=1.8,
        lataxis=dict(range=[-10, 65]),
        lonaxis=dict(range=[-30, 120]),
    ),
    margin=dict(l=10, r=10, t=50, b=10),
    height=650,
    width=1100,
    annotations=[dict(
        text='Source: Eurostat migr_eipre | Top 15 source nationalities by illegal entry volume',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.02, font=dict(size=10, color='grey'),
    )],
)

fig.show()

In [15]:
fig.write_html('presentation/migration_flows_eu.html', include_plotlyjs=True)
fig.write_image('presentation/migration_flows_eu.png', width=1200, height=700, scale=2)
print("Saved: presentation/migration_flows_eu.html + .png")

Saved: presentation/migration_flows_eu.html + .png


In [ ]:
# --- UK Illegal Entry Routes Flow Map (Data_IER_D01) ---
import pandas as pd
import plotly.graph_objects as go

df_ier = pd.read_excel('data/illegal-entry-routes-to-the-uk-dataset-dec-2025.xlsx',
                       sheet_name='Data_IER_D01', header=1)

# Nationality → (lat, lon) lookup
NAT_COORDS = {
    'Iran': (32.4, 53.7),        'Afghanistan': (33.9, 67.7),
    'Iraq': (33.2, 43.7),        'Eritrea': (15.2, 39.8),
    'Albania': (41.2, 20.2),     'Syria': (35.0, 38.0),
    'Sudan': (12.9, 30.2),       'Vietnam': (14.1, 108.3),
    'Turkey': (39.9, 32.9),      'Somalia': (5.2, 46.2),
    'Ethiopia': (9.1, 40.5),     'Egypt': (26.8, 30.8),
    'Yemen': (15.6, 48.5),       'India': (20.6, 78.9),
    'Kuwait': (29.3, 47.6),      'Georgia': (42.3, 43.4),
    'Libya': (26.3, 17.2),       'Pakistan': (30.4, 69.3),
    'Nigeria': (9.1, 8.7),       'China': (35.9, 104.2),
    'Bangladesh': (23.7, 90.4),  'Sri Lanka': (7.9, 80.8),
}

# Nationality → ISO-2 for choropleth
NAT_ISO = {
    'Iran': 'IRN',        'Afghanistan': 'AFG',    'Iraq': 'IRQ',
    'Eritrea': 'ERI',     'Albania': 'ALB',        'Syria': 'SYR',
    'Sudan': 'SDN',       'Vietnam': 'VNM',        'Turkey': 'TUR',
    'Somalia': 'SOM',     'Ethiopia': 'ETH',       'Egypt': 'EGY',
    'Yemen': 'YEM',       'India': 'IND',          'Kuwait': 'KWT',
    'Georgia': 'GEO',     'Libya': 'LBY',          'Pakistan': 'PAK',
    'Nigeria': 'NGA',     'China': 'CHN',
    'Bangladesh': 'BGD',  'Sri Lanka': 'LKA',
}

# Aggregate: all illegal entry routes, all years, top 15 mapped nationalities
totals = df_ier.groupby('Nationality')['Number of detections'].sum().reset_index()
totals.columns = ['Nationality', 'value']
totals = totals[totals['Nationality'].isin(NAT_COORDS)]
totals = totals.sort_values('value', ascending=False).head(15)

DST_LAT, DST_LON = 51.5, -1.5  # UK

fig = go.Figure()

# Choropleth underlay
fig.add_trace(go.Choropleth(
    locations=[NAT_ISO[n] for n in totals['Nationality']],
    z=totals['value'],
    colorscale='Reds',
    showscale=True,
    colorbar=dict(
        title=dict(text='Detections<br>(2018–25)', font=dict(size=11)),
        len=0.5, y=0.3, thickness=12,
    ),
    marker_line_color='#555',
    marker_line_width=0.5,
    hovertemplate='%{text}<br>%{z:,.0f} detections<extra></extra>',
    text=totals['Nationality'].values,
))

max_val = totals['value'].max()

for _, row in totals.iterrows():
    nat = row['Nationality']
    src_lat, src_lon = NAT_COORDS[nat]
    value = row['value']
    width = max(1.0, 8 * (value / max_val))
    opacity = max(0.3, 0.9 * (value / max_val))

    fig.add_trace(go.Scattergeo(
        lon=[src_lon, DST_LON],
        lat=[src_lat, DST_LAT],
        mode='lines',
        line=dict(width=width, color=f'rgba(0, 50, 120, {opacity})'),
        hoverinfo='text',
        text=f"{nat}: {value:,.0f}",
        showlegend=False,
    ))

    label = nat if value > 5000 else ''
    fig.add_trace(go.Scattergeo(
        lon=[src_lon], lat=[src_lat],
        mode='markers+text',
        marker=dict(size=max(5, 14 * (value / max_val)), color='#c0392b', opacity=0.85),
        text=label,
        textposition='top center',
        textfont=dict(size=9, color='#222'),
        hoverinfo='text',
        hovertext=f"{nat}<br>{value:,.0f} detected arrivals (2018–2025)",
        showlegend=False,
    ))

# UK destination marker
fig.add_trace(go.Scattergeo(
    lon=[DST_LON], lat=[DST_LAT],
    mode='markers+text',
    marker=dict(size=18, color='#003078', symbol='star',
                line=dict(width=1.5, color='white')),
    text=['UK'],
    textposition='bottom center',
    textfont=dict(size=12, color='#003078', family='Arial Black'),
    hoverinfo='text',
    hovertext='United Kingdom',
    showlegend=False,
))

fig.update_layout(
    title=dict(
        text='Detected Arrivals via Illegal Routes to the UK by Nationality<br>'
             '<sup>All entry methods combined, 2018 Q1 – 2025 Q4</sup>',
        font=dict(size=17, family='Arial'),
        x=0.5,
    ),
    geo=dict(
        showcountries=True,
        countrycolor='#ccc',
        showland=True,
        landcolor='#f4f4f4',
        showocean=True,
        oceancolor='#e6eef7',
        showlakes=True,
        lakecolor='#e6eef7',
        projection_type='natural earth',
        center=dict(lat=30, lon=35),
        projection_scale=1.8,
        lataxis=dict(range=[-5, 65]),
        lonaxis=dict(range=[-25, 115]),
    ),
    margin=dict(l=10, r=10, t=65, b=10),
    height=650,
    width=1100,
    annotations=[dict(
        text='Source: Home Office – Illegal entry routes to the UK dataset, Dec 2025 (Data_IER_D01)',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.02, font=dict(size=10, color='grey'),
    )],
)

fig.show()

In [ ]:
fig.write_html('presentation/migration_flows_uk.html', include_plotlyjs=True)
fig.write_image('presentation/migration_flows_uk.png', width=1200, height=700, scale=2)
print("Saved: presentation/migration_flows_uk.html + .png")